In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()

for candidate in [cwd, *cwd.parents]:
    if (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find the project root containing the 'data' folder.")

DATA_DIR = PROJECT_ROOT / "data"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: /home/rafidahmed/Coding/AI /rag_pipeline
DATA_DIR: /home/rafidahmed/Coding/AI /rag_pipeline/data


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/home/rafidahmed/Coding/AI /rag_pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Searching PDFs in: {pdf_dir}")
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data/pdf directory
all_pdf_documents = process_all_pdfs(DATA_DIR / "pdf")
print(f"all_pdf_documents contains {len(all_pdf_documents)} documents")

Searching PDFs in: /home/rafidahmed/Coding/AI /rag_pipeline/data/pdf
Found 1 PDF files to process

Processing: dp_presentation.pdf
  Loaded 19 pages

Total documents loaded: 19
all_pdf_documents contains 19 documents


In [4]:
# split document into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],)
    
    all_chunks = []
    
    for doc in documents:
        try:
            chunks = text_splitter.split_text(doc.page_content)
            for i, chunk in enumerate(chunks):
                chunk_doc = doc.copy()
                chunk_doc.page_content = chunk
                chunk_doc.metadata['chunk_index'] = i
                all_chunks.append(chunk_doc)
            print(f"  ✓ Split document into {len(chunks)} chunks")
        except Exception as e:
            print(f"  ✗ Error splitting document: {e}")
    
    print(f"\nTotal chunks created: {len(all_chunks)}")
    return all_chunks

In [5]:
chunks = split_documents(all_pdf_documents)

  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks
  ✓ Split document into 1 chunks

Total chunks created: 19


/tmp/ipykernel_9151/2937743936.py:16: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  chunk_doc = doc.copy()


In [6]:
# Show example chunks
print("Example chunks:")
for i, chunk in enumerate(chunks[:5]):  # Show first 5 chunks
    print(f"\nChunk {i+1} (from {chunk.metadata.get('source_file', 'unknown')}):")
    print(chunk.page_content[:300] + "..." if len(chunk.page_content) > 300 else chunk.page_content)
    print("-" * 50)

Example chunks:

Chunk 1 (from dp_presentation.pdf):
Introduction 
Many modern apps and speech-based systems collect
and process users’ voice data.
This voice data often contains personal or sensitive
information.
Without proper protection, attackers can misuse it to
steal identities or gain unauthorized access.
3
--------------------------------------------------

Chunk 2 (from dp_presentation.pdf):
Domain Introduction
Audio Anonymization:
Hides or masks the
speaker’s identity while
keeping the spoken
content clear and
understandable.
Audio Encryption:
Scrambles the entire
audio so it becomes
completely unreadable
without decryption.
Use Case: 
Enables safe sharing of
speech data for daily life...
--------------------------------------------------

Chunk 3 (from dp_presentation.pdf):
Literature
Review
5
--------------------------------------------------

Chunk 4 (from dp_presentation.pdf):
The VoicePrivacy 2022 Challenge: Progress &
Perspectives in Voice Anonymisation
Authors: Pierre 

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import uuid
import chromadb
from chromadb.config import Settings
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
class EmbeddingManager:
    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"✓ Loaded model: {self.model_name} with embedding dimension {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"✗ Error loading model '{self.model_name}': {e}")
            
    
    def create_embeddings(self, texts: List[str]) -> np.ndarray:
        """Create embeddings for a list of texts"""
        try:
            print(f"Creating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"✓ Created embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"✗ Error creating embeddings: {e}")
            return np.array([])
        
        

In [9]:
embedding_manager = EmbeddingManager(
    model_name="all-MiniLM-L6-v2",
)

Loading model: all-MiniLM-L6-v2
✓ Loaded model: all-MiniLM-L6-v2 with embedding dimension 384


In [20]:
class VectorStore:
    def __init__(self, collection_name: str="pdf_documents", persist_directory: str="../../vector_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection"""
        try:
            print(f"Initializing ChromaDB client with persist directory: {self.persist_directory}")
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            existing = [col.name for col in self.client.list_collections()]
            if self.collection_name in existing:
                self.collection = self.client.get_collection(name=self.collection_name)
                print(f"✓ Loaded existing collection: {self.collection_name}")
            else:
                self.collection = self.client.create_collection(name=self.collection_name)
                print(f"✓ Created new collection: {self.collection_name}")
        except Exception as e:
            print(f"✗ Error initializing ChromaDB: {e}")
            
            
    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        """Add documents and their embedding to the vector store"""
        try:
            ids = [str(uuid.uuid4()) for _ in documents]
            texts = [doc.page_content for doc in documents]          # <-- fix here
            metadatas = [doc.metadata for doc in documents]
            embeddings = embedding_manager.create_embeddings(texts)
            self.collection.add(ids=ids, embeddings=embeddings, metadatas=metadatas, documents=texts)
            print(f"✓ Added {len(documents)} documents to collection '{self.collection_name}'")
        except Exception as e:
            print(f"✗ Error adding documents: {e}")
            
 

In [21]:
vectorstore = VectorStore(
    collection_name="pdf_documents"
)  

Initializing ChromaDB client with persist directory: ../../vector_db
✓ Loaded existing collection: pdf_documents


In [22]:
chunks

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-10-28T12:02:30+00:00', 'title': 'DP presentation', 'moddate': '2025-10-28T12:02:29+00:00', 'keywords': 'DAG2hyVnXFs,BAGndFZHqL4,0', 'author': 'Sadman Sahil', 'source': '/home/rafidahmed/Coding/AI /rag_pipeline/data/pdf/dp_presentation.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1', 'source_file': 'dp_presentation.pdf', 'file_type': 'pdf', 'chunk_index': 0}, page_content='Introduction \nMany modern apps and speech-based systems collect\nand process users’ voice data.\nThis voice data often contains personal or sensitive\ninformation.\nWithout proper protection, attackers can misuse it to\nsteal identities or gain unauthorized access.\n3'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-10-28T12:02:30+00:00', 'title': 'DP presentation', 'moddate': '2025-10-28T12:02:29+00:00', 'keywords': 'DAG2hyVnXFs,BAGndFZHqL4,0', 'author': 'Sadman Sahil', 'source': '/home/rafid

In [19]:
# Convert text to embeddings 
texts = [chunk.page_content for chunk in chunks]
texts

['Introduction \nMany modern apps and speech-based systems collect\nand process users’ voice data.\nThis voice data often contains personal or sensitive\ninformation.\nWithout proper protection, attackers can misuse it to\nsteal identities or gain unauthorized access.\n3',
 'Domain Introduction\nAudio Anonymization:\nHides or masks the\nspeaker’s identity while\nkeeping the spoken\ncontent clear and\nunderstandable.\nAudio Encryption:\nScrambles the entire\naudio so it becomes\ncompletely unreadable\nwithout decryption.\nUse Case: \nEnables safe sharing of\nspeech data for daily life\nusage, research or AI\nconversion without\nrevealing who spoke.\n4',
 'Literature\nReview\n5',
 'The VoicePrivacy 2022 Challenge: Progress &\nPerspectives in Voice Anonymisation\nAuthors: Pierre Champion, Nicholas Evans, Sarina Meyer, Xiaoxiao Miao, Michele\nPanariello, Massimiliano Todisco, Natalia Tomashenko, Emmanuel Vincent, Xin Wang,\nJunichi Yamagishi\nPublication Date: 16 July, 2024\nPublication: I

In [23]:
# Generate embeddings for the chunks
embeddings = embedding_manager.create_embeddings(texts)
print(f"Generated embeddings shape: {embeddings.shape}")

Creating embeddings for 19 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.74it/s]

✓ Created embeddings with shape: (19, 384)
Generated embeddings shape: (19, 384)


In [24]:
# Add chunks to vector store
vectorstore.add_documents(documents=chunks, embeddings=embeddings)

Creating embeddings for 19 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.62it/s]

✓ Created embeddings with shape: (19, 384)


✓ Added 19 documents to collection 'pdf_documents'
